# Разбор A. Область применимости

Модель выдаёт число для любой допустимой структуры, включая далёкую от обучения. Гипотеза: расстояние до train выявляет случаи, где уверенность не обоснована. На A проверяем истрадефиллин, B отдельно проверяет кислоту и лактон. Ни similarity threshold, ни узкий bootstrap интервал автоматически не создают экспериментальную валидацию.

Сохраните тот же identity split seed=42. Рассчитайте Morgan radius=2,2048 для train и двух запросов; выведите ближайшего соседа и его измеренный logS. Совпадение структуры обработайте отдельно. Сравните ближайшее сходство кислоты и лактона B и обсудите, что отсутствует в наборе условий.

In [ ]:
from pathlib import Path
import sys,json,math
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'programm/course.json').is_file())
sys.path.insert(0,str(ROOT/'practicuum/shared'))
from course_support import DATA,compound,esol,workspace,write_json,init_dossier
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
def publish_plot(fig,name):
    fig.savefig(OUT/name,dpi=120,bbox_inches='tight')
    display(fig)
    plt.close(fig)
OUT=workspace('M12-L02','case')


Сначала проверяем входы и единицы. Словари сохраняют названия полей вместе со значениями; список сохраняет соответствие записей. Выведите промежуточный результат перед дальнейшим анализом.

In [ ]:
from rdkit import Chem,DataStructs
from rdkit.Chem import rdFingerprintGenerator
from sklearn.model_selection import GroupShuffleSplit
import numpy as np
rows=esol();mols=[Chem.MolFromSmiles(r['smiles']) for r in rows]
groups=[Chem.MolToSmiles(m,isomericSmiles=True) for m in mols]
train,test=next(GroupShuffleSplit(n_splits=1,test_size=.2,random_state=42).split(rows,groups=groups))
gen=rdFingerprintGenerator.GetMorganGenerator(radius=2,fpSize=2048,includeChirality=True)
fps=[gen.GetFingerprint(mols[i]) for i in train]


Теперь сравним модель с контрольным случаем. `assert` останавливает расчёт при нарушении контракта; это защита от ошибочного продолжения, а не оценка научного понимания.

In [ ]:
nearest=[]
for name in ['istradefylline','atorvastatin','atorvastatin_lactone']:
    query=gen.GetFingerprint(Chem.MolFromSmiles(compound(name)['smiles']))
    similarities=DataStructs.BulkTanimotoSimilarity(query,fps);j=int(np.argmax(similarities));idx=int(train[j])
    nearest.append({'name':name,'similarity':similarities[j],'neighbor_row_id':idx,'neighbor':rows[idx]['name'],
                    'neighbor_measured_logS':rows[idx]['measured'],'status':'REVIEW: threshold uncalibrated'})
display(nearest)


Сохраним именно полученный результат. Не переносите этот вывод на другую химическую форму, мишень или экспериментальные условия без отдельной проверки.

In [ ]:
result={'kind':'calculated','radius':2,'fpSize':2048,'nearest':nearest,'B_measured_solubility':'NOT_RUN'}


In [ ]:
write_json(OUT/'result.json',result)
from dossier_records import append_record
receipt=append_record(init_dossier(),'M12-L02','A',result,ROOT,inputs=['practicuum/shared/data/esol.csv', 'practicuum/shared/data/compounds.json'])
print('Versioned evidence:',receipt)


Ближайший сосед помогает диагностике, но его измерение не становится измерением query. Следующий шаг научного проекта — не только прогноз свойства: потребуется реализуемый синтетический маршрут и технологическая оценка.

Перед переходом ответьте: какие результаты получены вычислением, какие заданы условно, и какая независимая проверка ещё отсутствует?